# FastViT → ADE20K semantic segmentation (Colab)

Fine-tunes an ImageNet-pretrained FastViT backbone + Semantic FPN decoder on ADE20K (150 classes),
following Sec. 4.4 / Table 9 of the FastViT paper.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (or better).

Checkpoints and logs are written to Google Drive, so a disconnect doesn't lose progress.
Re-run all cells after a disconnect and training resumes from `last.pth` automatically.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive (stores checkpoints across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the code and install dependencies

In [ ]:
%cd /content
!rm -rf ml-fastvit
!git clone -q https://github.com/Sanugiw/ml-fastvit.git
%cd /content/ml-fastvit
!pip -q install timm datasets

## 4. Settings

| Model | Paper mIoU (Table 9) |
|---|---|
| fastvit_sa12 | 38.0 |
| fastvit_sa24 | 41.0 |
| fastvit_sa36 | 42.9 |
| fastvit_ma36 | 44.6 |

The paper schedule is 40k iterations at batch 32. On one T4 that's far more than a free Colab session.
`ITERS = 8000` is a realistic assignment budget (spread it over several sessions if needed);
report the shortened schedule in your write-up.

`CHECKPOINT`: leave as `None` to download the README's **unfused** ImageNet checkpoint automatically,
or point it at your own copy on Drive, e.g. `'/content/drive/MyDrive/fastvit/fastvit_sa12.pth.tar'`.
Do **not** use a `*_reparam.pth.tar` file.

In [ ]:
MODEL = 'fastvit_sa12'
ITERS = 8000
BATCH_SIZE = 8      # lower to 4 if you get CUDA out-of-memory (then set ACCUM_STEPS = 8)
ACCUM_STEPS = 4     # effective batch = BATCH_SIZE * ACCUM_STEPS = 32, as in the paper
CHECKPOINT = None   # None = auto-download unfused ImageNet weights
OUTPUT = f'/content/drive/MyDrive/fastvit_seg/{MODEL}'

import os
os.makedirs(OUTPUT, exist_ok=True)
ckpt_arg = f'--checkpoint "{CHECKPOINT}"' if CHECKPOINT else ''
resume_arg = f'--resume "{OUTPUT}/last.pth"' if os.path.exists(f'{OUTPUT}/last.pth') else ''
print('resuming' if resume_arg else 'starting fresh', '->', OUTPUT)

## 5. (Optional) 2-minute sanity check

Runs 20 iterations on 64 images to confirm everything works before the long run.
The first run also downloads ADE20K (~1 GB) into the Colab disk.

In [ ]:
!python segmentation/train_seg.py --model {MODEL} {ckpt_arg} --iters 20 --batch-size 2 --accum-steps 1     --train-samples 64 --eval-interval 20 --eval-samples 10 --log-interval 5 --workers 2 --amp     --output /content/debug_run

## 6. Train

Saves `last.pth` every 500 iterations and `best.pth` whenever validation mIoU improves
(on 500 val images during training; the full 2,000 at the end).
If Colab disconnects: reconnect, re-run cells 2–4, then this cell. It resumes automatically.

In [ ]:
!python segmentation/train_seg.py --model {MODEL} {ckpt_arg} {resume_arg}     --iters {ITERS} --batch-size {BATCH_SIZE} --accum-steps {ACCUM_STEPS} --amp     --eval-interval 2000 --eval-samples 500 --save-interval 500 --workers 2     --output "{OUTPUT}"

## 7. Training curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f'{OUTPUT}/log.csv')
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
t = log.dropna(subset=['loss']); ax[0].plot(t['iter'], t['loss']); ax[0].set(title='train loss', xlabel='iteration')
e = log.dropna(subset=['mIoU']); ax[1].plot(e['iter'], e['mIoU'], 'o-'); ax[1].set(title='val mIoU (%)', xlabel='iteration')
plt.tight_layout(); plt.savefig(f'{OUTPUT}/curves.png', dpi=150); plt.show()

## 8. Evaluate on the full val set + structural reparameterization

Evaluates the trained (multi-branch) model and the reparameterized (fused) model.
mIoU should be identical, while the fused model is faster. That's the paper's central idea.

In [ ]:
!python segmentation/eval_seg.py --checkpoint "{OUTPUT}/best.pth" --reparam --benchmark --vis 8

## 9. Look at predictions (image | ground truth | prediction)

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{OUTPUT}/vis/*.png'))[:8]:
    display(Image(p, width=900))

In [ ]:
import json
r = json.load(open(f'{OUTPUT}/eval_results.json'))
print({k: round(r['unfused'][k], 2) for k in ('mIoU', 'aAcc', 'mAcc')}, 'fused:', {k: round(r['fused'][k], 2) for k in ('mIoU',)})
print('latency (ms):', r['latency_ms'])